# c4fairness — worked example (regression)

Same flow as the binary example, but the model outputs a **continuous** prediction,
so the error is the signed residual `y_true - y_pred` and each cluster is summarised
by its **median error** (with one-way ANOVA / Mann-Whitney significance). Sensitive
kinds again: **binary** `gender`, **multi-categorical** `region`, **numeric** `age`.

In [ ]:
import pandas as pd
from c4f.preprocessing import encode_categoricals
from c4f.clustering import cluster
from c4f.cli import _build_sensitive_analysis_list, apply_salient_reconstruction
from c4f.experiments import make_recap
from c4f.result_viz import plot_cluster_recap_heatmap
from IPython.display import Image

df = pd.read_csv("data/example.csv")
df[["feat1", "feat2", "y_true_reg", "y_pred_reg", "gender", "region", "age"]].head()

## 1. Columns + encode + cluster (identical setup)

In [ ]:
regular   = ["feat1", "feat2"]
sensitive = ["gender", "region", "age"]
col_lists = {"regular": regular, "sensitive": sensitive, "proxy": [], "special": []}
orig_sensitive = list(sensitive)

dfe, cl, cat_names, mcd, ohe = encode_categoricals(
    df.copy(), col_lists, [], "kmeans", distance="euclidean"
)
clustering_cols = cl["regular"] + cl["sensitive"]
res = cluster(dfe[clustering_cols], algorithm="kmeans", distance="euclidean",
              n_clusters=3, random_state=42)
print("clusters:", res.n_clusters, "| silhouette:", round(res.silhouette, 3))

## 2. Continuous error + recap

The error column is the signed residual; `error_type="regression"` makes the recap
report each cluster's median error (`error_mean` / `abs_error_mean`) and use ANOVA /
Mann-Whitney for significance instead of Fisher.

In [ ]:
analysis = _build_sensitive_analysis_list(cl["sensitive"], mcd, orig_sensitive, option="salient")

dfe["residual"] = (df["y_true_reg"] - df["y_pred_reg"]).values
res_df = dfe.copy()
res_df["clusters"] = res.labels
apply_salient_reconstruction(res_df, mcd, orig_sensitive)

recap = make_recap(res_df, clustering_cols, sensitive_cols=analysis,
                   error_col="residual", error_type="regression",
                   feature_matrix=res.feature_matrix,
                   continuous_sensitive_cols=["age"])
recap.round(3)

## 3. Heatmap

In [ ]:
plot_cluster_recap_heatmap(recap.copy(), "regression_example", ".", error_label="residual")
Image("regression_example.png")

## Takeaway

`error_mean` is each cluster's average signed residual (bias direction) and
`abs_error_mean` its magnitude. A cluster with a large `|error| mean` and a
significant `error gap sig.` is where predictions are least reliable; its sensitive
columns show which group bears that error.